In [ ]:
!pip install jpegio

import os, torch, cv2, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import jpegio as jio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.ndimage import gaussian_filter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

In [ ]:
import shutil
import os

working_dir = '/kaggle/working'
for item in os.listdir(working_dir):
    item_path = os.path.join(working_dir, item)
    try:
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.unlink(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
        print(f" Deleted: {item}")
    except Exception as e:
        print(f" Failed to delete {item_path}: {e}")

print(" Working directory clear!")

In [ ]:
import requests
import tarfile
from tqdm import tqdm

TAR_URL = "https://huggingface.co/datasets/plok5308/djpeg_dataset/resolve/main/jpeg_data.tar.gz"
EXTRACT_DIR = '/kaggle/working/extracted'

os.makedirs(os.path.join(EXTRACT_DIR, 'single'), exist_ok=True)
os.makedirs(os.path.join(EXTRACT_DIR, 'double'), exist_ok=True)

def stream_extract(url, extract_dir, limit=40000):
    s_count, d_count = 0, 0
    print("Streaming extraction")

    response = requests.get(url, stream=True)
    # Stream mode 'r|gz' zaroori hai pipe-like extraction ke liye
    with tarfile.open(fileobj=response.raw, mode="r|gz") as tar:
        for member in tqdm(tar, desc="Processing"):
            if member.isfile() and member.name.lower().endswith('.jpg'):
                label = None
                if 'single' in member.name.lower() and s_count < limit:
                    label = 'single'
                    s_count += 1
                elif 'double' in member.name.lower() and d_count < limit:
                    label = 'double'
                    d_count += 1

                if label:
                    member.name = os.path.basename(member.name)
                    tar.extract(member, path=os.path.join(extract_dir, label))

            if s_count >= limit and d_count >= limit:
                break
    print(f"Extraction Done: Single={s_count}, Double={d_count}")

stream_extract(TAR_URL, EXTRACT_DIR)

In [ ]:
import os, torch, cv2, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import jpegio as jio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.ndimage import gaussian_filter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DenseNet-121 Architecture: Growth rate k=32 [cite: 12, 219]
class DenseLayer(nn.Module):
    def __init__(self, in_ch, k):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, 4*k, 1, bias=False),
            nn.BatchNorm2d(4*k), nn.ReLU(inplace=True),
            nn.Conv2d(4*k, k, 3, padding=1, bias=False)
        )
    def forward(self, x):
        return torch.cat([x, self.net(x)], 1)

class DenseNet121_JPEG(nn.Module):
    def __init__(self, num_classes=2, k=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        in_ch, config = 64, [6, 12, 24, 16]
        self.blocks = nn.ModuleList()
        for i, nl in enumerate(config):
            block = nn.Sequential(*[DenseLayer(in_ch + j*k, k) for j in range(nl)])
            self.blocks.append(block)
            in_ch += nl * k
            if i < 3:
                trans = nn.Sequential(
                    nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
                    nn.Conv2d(in_ch, in_ch//2, 1, bias=False),
                    nn.AvgPool2d(2, stride=2)
                )
                self.blocks.append(trans)
                in_ch //= 2
        self.final_bn = nn.BatchNorm2d(in_ch)
        self.classifier = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stem(x)
        for b in self.blocks: x = b(x)
        x = F.relu(self.final_bn(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.classifier(x)

In [ ]:
def extract_features(path, patch_coords=None):
    try:
        jpg = jio.read(path)
        # 128x128 patch extraction (changed from 256x256)
        if patch_coords:
            y, x = patch_coords
            dct = jpg.coef_arrays[0][y:y+128, x:x+128]
        else:
            dct = jpg.coef_arrays[0][:128, :128]

        q = jpg.quant_tables[0].reshape(8,8) if jpg.quant_tables[0].ndim==1 else jpg.quant_tables[0]
        if dct.shape != (128, 128): return None

        b = 100 # Histogram range [-100, 100] [cite: 285, 329]
        bins = np.arange(-b, b+2) - 0.5
        H = np.zeros((64, 2*b+1), dtype=np.float32)
        Q = np.zeros((64, 2*b+1), dtype=np.float32)

        for idx, (u, v) in enumerate((u,v) for u in range(8) for v in range(8)):
            c = np.clip(dct[u::8, v::8].flatten(), -b, b)
            H[idx], _ = np.histogram(c, bins=bins)
            Q[idx] = float(q[u,v]) # Repeated q-factors [cite: 178]

        X = np.stack([H, Q], axis=-1) # Dimension 64 x 201 x 2 [cite: 185]
        X /= (X.max() + 1e-6)
        return X
    except: return None

class JPEGDataset(Dataset):
    def __init__(self, root_dir):
        self.file_list = []
        for label in ['single', 'double']:
            path = os.path.join(root_dir, label)
            if os.path.exists(path):
                for f in os.listdir(path):
                    self.file_list.append((os.path.join(path, f), 0 if label=='single' else 1))

    def __len__(self): return len(self.file_list)
    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        X = extract_features(path)
        if X is None: return torch.zeros((2, 64, 201)), label
        return torch.tensor(X).permute(2, 0, 1), label

In [ ]:
# Setup
train_loader = DataLoader(JPEGDataset('/kaggle/working/extracted'), batch_size=64, shuffle=True)
model = DenseNet121_JPEG().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

print("Training Start (5 Epochs)...")
for epoch in range(5):
    model.train()
    tp, tn, fp, fn = 0, 0, 0, 0 # Confusion Matrix components [cite: 301, 302]

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)

        # Calculate Metrics [cite: 301, 302]
        tp += ((preds == 1) & (labels == 1)).sum().item() # Correct Double
        tn += ((preds == 0) & (labels == 0)).sum().item() # Correct Single
        fp += ((preds == 1) & (labels == 0)).sum().item() # Incorrect Double
        fn += ((preds == 0) & (labels == 1)).sum().item() # Incorrect Single

    # Paper Formulas [cite: 293, 295, 296]
    acc = (tp + tn) / (tp + tn + fp + fn + 1e-7)
    tpr = tp / (tp + fn + 1e-7)
    tnr = tn / (tn + fp + 1e-7)

    print(f"Epoch {epoch+1} | Acc: {acc*100:.2f}% | TPR: {tpr*100:.2f}% | TNR: {tnr*100:.2f}%")
    torch.save(model.state_dict(), '/kaggle/working/best_model.pth')

In [ ]:
import os
print("🔎 Scanning every corner for your .pth file...")
!find / -name "*.pth" 2>/dev/null